# 语义内核工具使用示例

本文档概述并解释了用于创建基于语义内核工具的代码，该工具与 ChromaDB 集成以实现检索增强生成（RAG）。此示例展示了如何构建一个 AI 代理，从 ChromaDB 集合中检索旅行文档，用语义搜索结果增强用户查询，并流式传输详细的旅行推荐内容。


SQLite版本修复  
如果您遇到以下错误：  
```
RuntimeError: Your system has an unsupported version of sqlite3. Chroma requires sqlite3 >= 3.35.0
```  

在笔记本开头取消注释以下代码块：  


In [ ]:
# %pip install pysqlite3-binary
# import sys
# sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

### 导入包
以下代码导入了必要的包：


In [2]:
!pip install "chromadb~=0.6.3"

  Using cached opentelemetry_sdk-1.39.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_api-1.39.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_semantic_conventions-0.60b1-py3-none-any.whl.metadata (2.4 kB)
INFO: pip is looking at multiple versions of opentelemetry-instrumentation-fastapi to determine which version is compatible with other requirements. This could take a while.
  Using cached opentelemetry_instrumentation-0.62b0-py3-none-any.whl.metadata (7.2 kB)
  Using cached opentelemetry_instrumentation-0.61b0-py3-none-any.whl.metadata (7.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 280.0 kB/s  0:00:02eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 279.7 kB/s  0:00:07 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.7/17.7 MB 264.6 kB/s  0:01:06m0:00:0100:03
Using cached opentelemetry_sdk-1.39.1-py3-none-any.whl (132 kB)
Using cached opentelemetry_api-1.39.1-py3-none-any.whl (66 kB)
Using cached ope

In [9]:
# 导入JSON处理库，用于处理JSON格式数据
import json
# 导入操作系统接口库，用于访问环境变量和文件系统
import os
# 从dotenv库导入load_dotenv函数，用于加载.env文件中的环境变量
from dotenv import load_dotenv
# 安装ChromaDB（如果尚未安装）：pip install chromadb
import chromadb
# 从typing模块导入类型注解工具，用于给函数参数和返回值添加类型说明
from typing import Annotated, TYPE_CHECKING

# 从IPython显示模块导入display和HTML，用于在Jupyter Notebook中美化输出
from IPython.display import display, HTML

# 导入AsyncOpenAI，这是OpenAI的异步客户端，用于与大模型API通信
from openai import AsyncOpenAI

# 从语义内核(semantic kernel)库导入关键组件：
# - ChatCompletionAgent: 创建能够完成聊天任务的AI代理
# - ChatHistoryAgentThread: 管理对话历史的线程
from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
# OpenAIChatCompletion: 连接OpenAI兼容API的客户端
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
# 内容类型定义：
# - FunctionCallContent: 函数调用内容
# - FunctionResultContent: 函数执行结果内容
# - StreamingTextContent: 流式文本内容
from semantic_kernel.contents import FunctionCallContent, FunctionResultContent, StreamingTextContent
# kernel_function: 用于标记函数为可被AI调用的工具
from semantic_kernel.functions import kernel_function

# 仅在类型检查时使用的导入（不会在运行时执行）
if TYPE_CHECKING:
    # 导入ChromaDB集合类型，用于类型提示
    from chromadb.api.models.Collection import Collection

### 创建语义内核和 AI 服务

语义内核实例通过异步的 OpenAI 聊天完成服务创建并配置。该服务被添加到内核中，用于生成响应。


In [10]:
# 加载.env文件中的环境变量（如果存在）
# .env文件通常包含API密钥等敏感信息，不应提交到代码仓库
load_dotenv()

# 创建AsyncOpenAI客户端实例
# 这是一个异步客户端，可以非阻塞地与大模型API通信
# 下面两种大模型都可以试一下，效果不一样，千问有些情况并不会调用function call


# model_name="qwen-max"
# client = AsyncOpenAI(
#     # 从环境变量获取DashScope（阿里云通义千问）API密钥
#     api_key=os.environ.get("DASHSCOPE_API_KEY"), 
#     # 设置API基础URL为阿里云DashScope的兼容模式端点
#     base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
# )

model_name = "gpt-4.1-mini"
client = AsyncOpenAI(
    api_key=os.environ["GITHUB_TOKEN"],
    base_url="https://models.inference.ai.azure.com/"
)

# 创建AI服务对象，作为Semantic Kernel与Qwen-Max模型通信的桥梁
chat_completion_service = OpenAIChatCompletion(
    # 指定要使用的模型ID（这里是通义千问的qwen-max）
    ai_model_id=model_name,
    # 传入之前创建的AsyncOpenAI客户端
    async_client=client,
)

### 定义提示插件

PromptPlugin 是一个原生插件，它定义了一个函数，用于使用检索上下文构建增强提示


In [11]:
class PromptPlugin:
    """
    一个插件类，用于构建增强提示和检索上下文
    插件(Plugin)：Semantic Kernel中组织相关功能的类，可以包含多个可被AI调用的工具
    @kernel_function：装饰器，标记函数为可被AI调用的工具，并提供名称和描述
    RAG(检索增强生成)：先检索相关信息，再基于检索结果生成回答的技术
    ChromaDB：一个开源的向量数据库，用于存储和检索文本片段
    """
    
    def __init__(self, collection: "Collection"):
        """初始化插件，接收ChromaDB集合对象"""
        # 保存ChromaDB集合引用，用于后续检索
        self.collection = collection

    # 使用检索到的上下文构建增强提示
    @kernel_function(
        name="build_augmented_prompt",
        description="使用检索上下文构建增强提示。"
    )
    def build_augmented_prompt(self, query: str, retrieval_context: str) -> str:
        """
        构建增强提示的函数
        
        参数:
            query: 用户原始查询
            retrieval_context: 从向量数据库检索到的相关上下文
            
        返回:
            构建好的增强提示字符串
        """
        # 将检索到的上下文和用户查询组合成一个增强提示
        # 这个提示会告诉AI"基于以下上下文回答问题"
        return (
            f"检索到的上下文:\n{retrieval_context}\n\n"  # 检索到的上下文
            f"用户原始查询: {query}\n\n"  # 用户原始查询
            "仅基于上述上下文，请提供您的答案。"  # 指示AI仅基于上下文回答
        )
    
    # 从数据库检索上下文
    @kernel_function(name="retrieve_context", description="从数据库检索上下文。")
    def get_retrieval_context(self, query: str) -> str:
        """
        从ChromaDB检索与查询相关的上下文
        
        参数:
            query: 用户查询
            
        返回:
            格式化的检索结果字符串
        """
        # 使用ChromaDB查询与用户查询最相关的文档
        results = self.collection.query(
            query_texts=[query],  # 要查询的文本列表
            include=["documents", "metadatas"],  # 返回文档内容和元数据
            n_results=2  # 返回最相关的2个结果
        )
        
        # 准备存储格式化结果的列表
        context_entries = []
        
        # 检查是否有有效的检索结果
        if results and results.get("documents") and results["documents"][0]:
            # 将文档内容和元数据配对
            for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
                # 格式化每个结果条目
                context_entries.append(f"Document: {doc}\nMetadata: {meta}")
        
        print(f"context_entries: {context_entries}")
        # 如果有结果，用空行分隔返回；否则返回"未找到检索上下文"
        return self.build_augmented_prompt(query,"\n\n".join(context_entries)) if context_entries else "No retrieval context found."

### 定义天气信息插件

WeatherInfoPlugin 是一个原生插件，用于提供特定旅行目的地的温度信息。


In [13]:
class WeatherInfoPlugin:
    """提供旅行目的地平均温度信息的插件"""
    
    def __init__(self):
        # 创建一个字典，存储目的地和对应的平均温度
        self.destination_temperatures = {
            "马尔代夫": "82°F (28°C)",  # 马尔代夫
            "瑞士阿尔卑斯山": "45°F (7°C)",  # 瑞士阿尔卑斯山
            "非洲野生动物园": "75°F (24°C)"  # 非洲野生动物园
        }

    # Annotated：类型注解工具，可以为类型添加额外信息（这里是函数描述）
    # 获取特定旅行目的地的平均温度
    @kernel_function(description="获取特定旅行目的地的平均温度。")
    def get_destination_temperature(self, destination: str) -> Annotated[str, "返回目的地的平均温度"]:
        """
        获取指定旅行目的地的平均温度
        
        参数:
            destination: 旅行目的地名称
            
        返回:
            包含温度信息的字符串
        """
        # 将输入的目的地名称转换为小写，便于匹配
        normalized_destination = destination.lower()

        # 在字典中查找匹配的目的地
        if normalized_destination in self.destination_temperatures:
            # 找到匹配项，返回格式化的温度信息
            return f" {destination} 的平均温度是 {self.destination_temperatures[normalized_destination]}。"
        else:
            # 未找到匹配项，返回错误信息和可用目的地列表
            return f"抱歉，我没有关于 {destination} 的温度信息。可用的目的地有：马尔代夫、瑞士阿尔卑斯山和非洲野生动物园。"

### 定义目的地信息插件

DestinationsPlugin 是一个原生插件，提供有关热门旅行目的地的详细信息。


In [15]:
class DestinationsPlugin:
    """提供热门旅行目的地详细信息的插件"""

    # 存储目的地详细信息的字典
    # Destination data store with rich details about popular travel locations
    DESTINATIONS = {
        "马尔代夫": {
            "name": "马尔代夫",
            "description": "印度洋上的一个群岛国家，由26个环礁组成，以原始洁白的海滩、清澈的海水和水上别墅闻名。",
            "best_time": "11月至次年4月（旱季）",
            "activities": ["浮潜", "深潜", "跳岛游", "水疗度假", "海底餐厅用餐"],
            "avg_cost": "豪华度假村每晚约400-1200美元"
        },
        "瑞士阿尔卑斯山": {
            "name": "瑞士阿尔卑斯山",
            "description": "横跨瑞士的阿尔卑斯山脉，拥有风景如画的山间村庄和世界级滑雪胜地。",
            "best_time": "12月至3月适合滑雪，6月至9月适合徒步",
            "activities": ["滑雪", "单板滑雪", "徒步旅行", "山地自行车", "滑翔伞"],
            "avg_cost": "阿尔卑斯山区住宿每晚约250-500美元"
        },
        "非洲野生动物园": {
            "name": "非洲野生动物园",
            "description": "涵盖肯尼亚、坦桑尼亚和南非等多个非洲国家的野生动物观赏体验。",
            "best_time": "6月至10月（旱季），最适合观赏野生动物",
            "activities": ["越野观兽", "徒步探险", "热气球观光", "参观当地文化村"],
            "avg_cost": "豪华探险套餐每人每天约400-800美元"
        },
        "巴厘岛": {
            "name": "印度尼西亚·巴厘岛",
            "description": "以郁郁葱葱的梯田、美丽的寺庙和充满活力的当地文化而闻名的热带岛屿。",
            "best_time": "4月至10月（旱季）",
            "activities": ["冲浪", "参观寺庙", "梯田徒步", "瑜伽静修", "海滩休闲"],
            "avg_cost": "根据住宿类型不同，每晚约100-500美元"
        },
        "圣托里尼": {
            "name": "希腊·圣托里尼",
            "description": "风景壮丽的火山岛，以白墙蓝顶建筑和俯瞰爱琴海的绝美景色闻名。",
            "best_time": "4月下旬至11月初",
            "activities": ["伊亚观赏日落", "葡萄酒品鉴", "乘船游览", "海滩巡游", "探索古代遗址"],
            "avg_cost": "悬崖海景房每晚约200-600美元"
        }
    }
    # 提供特定旅行目的地的详细信息
    @kernel_function(
        name="get_destination_info",
        description="提供有关特定旅行目的地的详细信息。"
    )
    def get_destination_info(self, query: str) -> str:
        """
        根据查询获取目的地信息
        
        参数:
            query: 用户查询
            
        返回:
            格式化的目的地信息
        """
        # 将查询转换为小写以便匹配
        query_lower = query.lower()
        matching_destinations = []

        # 遍历所有目的地，查找与查询匹配的项
        for key, details in DestinationsPlugin.DESTINATIONS.items():
            # 检查目的地关键词或名称是否在查询中
            if key in query_lower or details["name"].lower() in query_lower:
                matching_destinations.append(details)
                
        if not matching_destinations:
            return (f"User Query: {query}\n\n"
                    f"我无法在数据库中找到特定的目的地信息。 "
                    f"请使用通用检索系统进行此查询。")

        # Format destination information
        destination_info = "\n\n".join([
            f"Destination: {dest['name']}\n"
            f"Description: {dest['description']}\n"
            f"Best time to visit: {dest['best_time']}\n"
            f"Popular activities: {', '.join(dest['activities'])}\n"
            f"Average cost: {dest['avg_cost']}" for dest in matching_destinations
        ])

        return (f"目地信息:\n{destination_info}\n\n"
                f"用户原始查询: {query}\n\n"
                "根据上述目的地详细信息，提供一个有用的响应，解决用户对该位置的查询。")

## 设置 ChromaDB

为了方便检索增强生成，需实例化一个持久化的 ChromaDB 客户端，并创建一个名为 `"travel_documents"` 的集合（如果已存在则直接检索）。然后将示例旅行文档和元数据填充到该集合中。


In [18]:
# 创建持久化的ChromaDB客户端
# 数据将存储在当前目录下的./chroma_db文件夹中
collection = chromadb.PersistentClient(path="./chroma_db").create_collection(
    name="travel_documents",  # 集合名称
    metadata={"description": "travel_service"},  # 集合元数据
    get_or_create=True,  # 如果集合已存在则获取它，否则创建新集合
)
# /Users/a1-6/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M
# ChromaDB 默认使用 all-MiniLM-L6-v2 这个轻量级模型来生成向量
"""
文档内容翻译
- **Contoso Travel** 提供前往全球异国目的地的豪华度假套餐。  
- 我们的高端旅行服务包括个性化行程规划和全天候24小时礼宾支持。  
- Contoso 的旅行保险涵盖医疗紧急情况、行程取消和行李丢失。  
- 受欢迎的目的地包括马尔代夫、瑞士阿尔卑斯山和非洲野生动物园。  
- Contoso Travel 提供独家进入精品酒店和私人导览服务。
"""
# 定义示例旅行文档（这些是将被存储和检索的文本）
documents = [
    "Contoso Travel提供前往全球异国目的地的豪华度假套餐",
    "我们的高端旅行服务包括个性化行程规划和全天候24小时礼宾支持。",
    "Contoso的旅行保险涵盖医疗紧急情况、行程取消和行李丢失。",
    "受欢迎的目的地包括马尔代夫、瑞士阿尔卑斯山和非洲野生动物园。",
    "Contoso Travel提供独家进入精品酒店和私人导览服务。",
]

# 将文档添加到集合中
# 在调用 .add() 时，只传了 documents 而没有传 embeddings 参数，ChromaDB 会自动调用它内置的 Embedding Function。将documents生成向量
collection.add(
    documents=documents,  # 要添加的文档列表
    ids=[f"doc_{i}" for i in range(len(documents))],  # 为每个文档生成唯一ID
    # 为每个文档添加元数据（来源和类型）
    metadatas=[{"source": "training", "type": "explanation"} for _ in documents]
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Add of existing embedding ID: doc_0
Add of existing embedding ID: doc_1
Add of existing embedding ID: doc_2
Add of existing embedding ID: doc_3
Add of existing embedding ID: doc_4
Insert of existing embedding ID: doc_0
Insert of existing embedding ID: doc_1
Insert of existing embedding ID: doc_2
Insert of existing embedding ID: doc_3
Insert of existing embedding ID: doc_4


In [20]:
# 创建聊天完成代理
agent = ChatCompletionAgent(
    service=chat_completion_service,  # 使用之前创建的AI服务
    # 注册三个插件，使代理能够调用这些插件中的工具
    plugins=[DestinationsPlugin(), WeatherInfoPlugin(), PromptPlugin(collection)],
    name="TravelAgent",  # 代理名称
    # 指令：告诉代理如何行为
    instructions="使用提供的上下文回答旅行查询，您需要首先检索上下文。如果提供了上下文，不要说’我没有上下文‘。",
)

### 运行代理并使用流式聊天记录
主要的异步循环会为对话创建一个聊天记录，并且对于每次用户输入，首先将增强的提示（作为系统消息）添加到聊天记录中，以便代理能够看到检索上下文。用户消息也会被添加，然后通过流式方式调用代理。输出会在流式传入时打印出来。


In [21]:
async def main():
    """主异步函数，处理用户查询和代理响应"""

    # 初始化对话线程（None表示新对话开始）
    thread: ChatHistoryAgentThread | None = None

    """
    你能介绍一下 Contoso 旅游保险的保障范围吗？
    马尔代夫的全年平均气温是多少？
    Contoso 有哪些推荐的寒冷旅游目的地？这些地方的平均气温是多少？
    """
    # 定义几个示例用户查询
    user_inputs = [
        "你能介绍一下 Contoso 旅游保险的保障范围吗？",
        "马尔代夫的全年平均气温是多少？",
        "Contoso 有哪些推荐的寒冷旅游目的地？这些地方的平均气温是多少？",
    ]

    # 遍历每个用户查询
    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        # 初始化变量用于收集响应
        agent_name = None  # 代理名称
        full_response: list[str] = []  # 完整响应文本
        function_calls: list[str] = []  # 记录调用的函数
        
        # 用于重建流式函数调用的缓冲区
        current_function_name = None  # 当前正在调用的函数名
        argument_buffer = ""  # 用于累积函数参数（流式传输时参数可能分块到达）

        # 异步遍历代理的流式响应
        async for response in agent.invoke_stream(
            messages=user_input,  # 传入用户查询
            thread=thread,  # 传入对话线程（用于维护对话历史）
        ):
            # 更新对话线程（可能被代理修改）
            thread = response.thread
            # 获取代理名称
            agent_name = response.name
            # 将响应项转换为列表以便处理
            content_items = list(response.items)

            # 遍历响应中的每个内容项
            for item in content_items:
                # 情况1：AI请求调用函数
                if isinstance(item, FunctionCallContent):
                    # 如果有函数名，记录当前调用的函数
                    if item.function_name:
                        current_function_name = item.function_name

                    # 累积函数参数（流式传输时参数可能分块到达）
                    if isinstance(item.arguments, str):
                        argument_buffer += item.arguments
                
                # 情况2：函数调用结果返回
                elif isinstance(item, FunctionResultContent):
                    # 处理之前累积的函数调用
                    if current_function_name:
                        # 清理累积的参数
                        formatted_args = argument_buffer.strip()
                        try:
                            # 尝试将参数解析为JSON（如果参数是JSON格式）
                            parsed_args = json.loads(formatted_args)
                            formatted_args = json.dumps(parsed_args)
                        except Exception:
                            pass  # 无法解析为JSON，保持原始字符串
                        
                        # 记录函数调用信息
                        function_calls.append(f"调用函数: {current_function_name}({formatted_args})")
                        # 重置函数名和参数缓冲区
                        current_function_name = None
                        argument_buffer = ""
                    
                    # 记录函数执行结果
                    function_calls.append(f"\n函数结果:\n\n{item.result}")
                
                # 情况3：流式文本响应
                elif isinstance(item, StreamingTextContent) and item.text:
                    # 将文本片段添加到完整响应中
                    full_response.append(item.text)


        # 如果有函数调用记录，添加到HTML输出
        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>"
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>Function Calls (click to expand)</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
                "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}"
                "</div></details></div>"
            )

        # 添加代理的最终响应到HTML输出
        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>{agent_name or 'Assistant'}:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        # 在Notebook中显示HTML输出
        display(HTML(html_output))

# 运行主函数
await main()


context_entries: ["Document: Contoso's travel insurance covers medical emergencies, trip cancellations, and lost baggage.\nMetadata: {'source': 'training', 'type': 'explanation'}", "Document: Contoso Travel provides exclusive access to boutique hotels and private guided tours.\nMetadata: {'source': 'training', 'type': 'explanation'}"]


context_entries: ["Document: Contoso Travel offers luxury vacation packages to exotic destinations worldwide.\nMetadata: {'source': 'training', 'type': 'explanation'}", "Document: Contoso Travel provides exclusive access to boutique hotels and private guided tours.\nMetadata: {'source': 'training', 'type': 'explanation'}"]



---

**免责声明**：  
本文档使用AI翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 进行翻译。尽管我们努力确保翻译的准确性，但请注意，自动翻译可能包含错误或不准确之处。原始语言的文档应被视为权威来源。对于关键信息，建议使用专业人工翻译。我们不对因使用此翻译而产生的任何误解或误读承担责任。
